# Cary NADH Kinetics Per-Sample Point Selector v11

This notebook is for Cary UV–Vis NADH kinetics exports at **340 nm**.

Workflow:

1. Load the Cary CSV file.
2. Plot **every sample** as its own separate graph.
3. For each sample graph, click the start and end point of the straight negative-slope region.
4. Enter the substrate concentration in **µM** beside each sample in the sheet-style table.
5. Check or uncheck whether each sample should be kept for analysis.
6. Samples with the same entered substrate concentration are treated as duplicates/triplicates/replicates.
7. The notebook calculates **sample-level raw replicate activity** plus a mean ± SD/SEM summary.
8. By default, kinetic models are fit to **raw replicate points**, not averages or SE bars.
9. Michaelis-Menten, Hill, and substrate-inhibition fits are all generated; no automatic winner is chosen.
9. Use the model toggle to compare **Michaelis-Menten**, **Hill**, and **substrate inhibition** fits.
10. The notebook estimates **nH** and **Ki** automatically from the Hill and substrate-inhibition curves.
11. Results are exported to Excel with raw fits, summary fits, model comparison, and fitted parameters.

Blank cells in the Cary file are not treated as real points, but **every sample column pair is still plotted/listed** if the file contains that sample.


In [ ]:
# Run this once if needed, then restart the kernel:
# %pip install pandas numpy scipy plotly ipywidgets anywidget openpyxl

In [1]:

# Cary NADH Kinetics Per-Sample Point Selector v11 — Two-Point Selector + Centralized Plot Settings
# =================================================
# Designed for Cary UV-Vis kinetics CSV exports where each sample is stored as:
#   Time(ms), Abs, Time(ms), Abs, ...
# at 340 nm.
#
# Workflow:
#   1. Load Cary CSV directly.
#   2. Every sample found in the Cary file gets its own graph.
#   3. Click start and end points on each graph to define the linear slope region.
#   4. Enter substrate concentration in µM in the sample table.
#   5. Check/uncheck which samples to keep for final analysis.
#   6. Samples with the same entered substrate concentration are treated as replicates.
#   7. Calculate sample-level raw replicate activities and display summaries by substrate concentration.
#   8. Calculate activity, specific activity, kcat, and fit Michaelis-Menten, Hill, or substrate-inhibition models.
#
# Install once if needed:
#   %pip install pandas numpy scipy plotly ipywidgets anywidget openpyxl

from pathlib import Path
import csv
import math
import numpy as np
import pandas as pd
from scipy.stats import linregress
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Default assay constants
# -------------------------
EXTINCTION_COEFF = 6.22       # NADH at 340 nm, mM^-1 cm^-1
PATH_LENGTH_CM = 1.0          # standard cuvette path length
REACTION_VOLUME_ML = 1.0      # total reaction volume; edit if not 1 mL
ENZYME_VOLUME_UL = 5.0        # enzyme volume added per reaction
SUBSTRATE_UNITS = "µM"
TIME_DISPLAY_UNITS = "min"


# -------------------------
# Plot appearance settings
# -------------------------
# Visual style based on the clean enzyme-kinetics figures in:
# Wang et al. (2020), Enzyme Kinetics by Isothermal Titration Calorimetry.
#
# Only appearance is changed here. Data, calculations, units, axis values,
# model fitting, selectors, exports, and notebook behavior remain unchanged.

PAPER_RED = "#FF0000"
PAPER_BLACK = "#000000"
PAPER_WHITE = "#FFFFFF"

PAPER_AXIS_STYLE = {
    "showline": True,
    "linewidth": 2.2,
    "linecolor": PAPER_BLACK,
    "mirror": False,
    "ticks": "outside",
    "tickwidth": 1.5,
    "ticklen": 6,
    "tickcolor": PAPER_BLACK,
    "showgrid": False,
    "zeroline": False,
    "tickfont": {
        "family": "Arial",
        "size": 14,
        "color": PAPER_BLACK,
    },
    "title_font": {
        "family": "Arial",
        "size": 18,
        "color": PAPER_BLACK,
    },
}

PAPER_GLOBAL_FONT = {
    "family": "Arial",
    "size": 14,
    "color": PAPER_BLACK,
}

SLOPE_SELECTOR_RAW_TRACE_STYLE = {
    "mode": "lines+markers",
    "name": "Raw A340 data",
    "line": {
        "color": PAPER_BLACK,
        "width": 1.3,
    },
    "marker": {
        "size": 4,
        "color": PAPER_BLACK,
        "line": {
            "width": 0.5,
            "color": PAPER_BLACK,
        },
    },
}

SLOPE_SELECTOR_EMPTY_RAW_TRACE_STYLE = {
    "mode": "lines+markers",
    "name": "Raw A340 data",
    "line": {
        "color": PAPER_BLACK,
        "width": 1.3,
    },
    "marker": {
        "size": 4,
        "color": PAPER_BLACK,
    },
}

SLOPE_SELECTOR_BOUNDARY_TRACE_STYLE = {
    "mode": "markers",
    "name": "Clicked start/end points",
    "marker": {
        "size": 11,
        "symbol": "circle-open",
        "color": PAPER_RED,
        "line": {
            "width": 2.2,
            "color": PAPER_RED,
        },
    },
}

SLOPE_SELECTOR_REGION_TRACE_STYLE = {
    "mode": "markers",
    "name": "Points used in fit",
    "marker": {
        "size": 6,
        "color": PAPER_BLACK,
        "line": {
            "width": 0.6,
            "color": PAPER_BLACK,
        },
    },
}

SLOPE_SELECTOR_FIT_TRACE_STYLE = {
    "mode": "lines",
    "name": "Slope fit",
    "line": {
        "color": PAPER_RED,
        "width": 2.5,
    },
}

SLOPE_SELECTOR_LAYOUT_STYLE = {
    "template": "plotly_white",
    "hovermode": "closest",
    "height": 520,
    "plot_bgcolor": PAPER_WHITE,
    "paper_bgcolor": PAPER_WHITE,
    "font": PAPER_GLOBAL_FONT,
    "title_font": {
        "family": "Arial",
        "size": 19,
        "color": PAPER_BLACK,
    },
    "xaxis": PAPER_AXIS_STYLE,
    "yaxis": PAPER_AXIS_STYLE,
    "legend": {
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
        "font": {
            "family": "Arial",
            "size": 12,
            "color": PAPER_BLACK,
        },
        "bgcolor": "rgba(255,255,255,0)",
        "borderwidth": 0,
    },
    "margin": {
        "l": 75,
        "r": 30,
        "t": 90,
        "b": 70,
    },
}

RAW_REPLICATE_MARKER_STYLE = {
    "size": 7,
    "color": PAPER_BLACK,
    "opacity": 1.0,
    "symbol": "circle",
    "line": {
        "width": 0.7,
        "color": PAPER_BLACK,
    },
}

MEAN_MARKER_STYLE = {
    "size": 10,
    "color": PAPER_WHITE,
    "symbol": "circle",
    "line": {
        "width": 2,
        "color": PAPER_BLACK,
    },
}

MEAN_ERROR_BAR_STYLE = {
    "type": "data",
    "visible": True,
    "color": PAPER_BLACK,
    "thickness": 1.5,
    "width": 4,
}

COMMON_PLOT_AXIS_STYLE = PAPER_AXIS_STYLE

COMMON_HORIZONTAL_LEGEND_STYLE = {
    "orientation": "h",
    "yanchor": "bottom",
    "y": 1.02,
    "xanchor": "center",
    "x": 0.5,
    "font": {
        "family": "Arial",
        "size": 12,
        "color": PAPER_BLACK,
    },
    "bgcolor": "rgba(255,255,255,0)",
    "borderwidth": 0,
}

ACTIVITY_PLOT_LAYOUT_STYLE = {
    "template": "plotly_white",
    "width": 900,
    "height": 620,
    "hovermode": "closest",
    "plot_bgcolor": PAPER_WHITE,
    "paper_bgcolor": PAPER_WHITE,
    "font": PAPER_GLOBAL_FONT,
    "title_font": {
        "family": "Arial",
        "size": 20,
        "color": PAPER_BLACK,
    },
    "legend": COMMON_HORIZONTAL_LEGEND_STYLE,
    "margin": {
        "l": 90,
        "r": 40,
        "t": 95,
        "b": 80,
    },
}

KINETIC_FIT_LINE_STYLE = {
    "color": PAPER_RED,
    "width": 2.7,
}

# Plain parameter text, matching the paper's uncluttered figure style.
KINETIC_PARAMETER_ANNOTATION_POSITION = {
    "x": 0.98,
    "y": 0.55,
    "xref": "paper",
    "yref": "paper",
}

KINETIC_PARAMETER_ANNOTATION_STYLE = {
    "showarrow": False,
    "align": "left",
    "xanchor": "right",
    "yanchor": "top",
    "bgcolor": "rgba(255,255,255,0)",
    "borderwidth": 0,
    "borderpad": 0,
    "font": {
        "family": "Arial",
        "size": 13,
        "color": PAPER_BLACK,
    },
}

KINETIC_PLOT_LAYOUT_STYLE = {
    "template": "plotly_white",
    "width": 950,
    "height": 650,
    "hovermode": "closest",
    "plot_bgcolor": PAPER_WHITE,
    "paper_bgcolor": PAPER_WHITE,
    "font": PAPER_GLOBAL_FONT,
    "title_font": {
        "family": "Arial",
        "size": 20,
        "color": PAPER_BLACK,
    },
    # Keep the numeric µM axis readable across both small and large ranges.
    # The fitted curve can still use hundreds of x-values; axis ticks are independent.
    "xaxis": {
        "tickmode": "auto",
        "nticks": 8,
        "tickformat": ".0f",
        "separatethousands": True,
    },
    "legend": COMMON_HORIZONTAL_LEGEND_STYLE,
    "margin": {
        "l": 90,
        "r": 45,
        "t": 95,
        "b": 80,
    },
}

# -------------------------
# File paths
# -------------------------
CARY_FILE = r"C:\Users\crumb\OneDrive\Desktop\kinetics notebook\rcPEPCK practice #4.csv"
OUTPUT_FILE = r"C:\Users\crumb\OneDrive\Desktop\kinetics notebook\cary_nadh_kinetics_results_v6_raw_model_toggle.xlsx"


def _to_float(value):
    """Safely convert CSV cell to float; return NaN for blanks/non-numeric cells."""
    if value is None:
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip().replace("\ufeff", "")
    if s == "":
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_cary_kinetics_csv(file_path):
    """
    Parse Cary UV-Vis kinetics CSV exports with repeating Time(ms), Abs pairs.

    Returns
    -------
    sample_info_df : DataFrame
        One row for every sample column pair found in the Cary file.
        Even very short/incomplete samples are included.

    long_df : DataFrame
        Long-format valid numeric time/absorbance coordinates.
        Blank rows cannot be plotted as points, but the sample itself remains in sample_info_df.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find file: {file_path}")

    with open(file_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        rows = list(csv.reader(f))

    # Find sample name row and data header row.
    name_row_idx = None
    header_row_idx = None
    for i, row in enumerate(rows):
        first = row[0].strip().replace("\ufeff", "") if row else ""
        if first == "Name":
            name_row_idx = i
        # Cary graph data header row typically starts with blank cell then Time(ms), Abs repeated.
        if any(str(cell).strip() == "Time(ms)" for cell in row) and any(str(cell).strip() == "Abs" for cell in row):
            header_row_idx = i
            break

    if name_row_idx is None:
        raise ValueError("Could not find Cary 'Name' row containing sample names.")
    if header_row_idx is None:
        raise ValueError("Could not find Cary graph data header row containing Time(ms), Abs pairs.")

    name_row = rows[name_row_idx]
    header_row = rows[header_row_idx]

    # Identify every Time(ms), Abs pair and assign sample names from the Name row.
    sample_pairs = []
    for c in range(len(header_row) - 1):
        if str(header_row[c]).strip() == "Time(ms)" and str(header_row[c+1]).strip() == "Abs":
            sample_name = name_row[c].strip() if c < len(name_row) and str(name_row[c]).strip() else f"Sample {len(sample_pairs)+1}"
            sample_pairs.append({
                "sample_index": len(sample_pairs) + 1,
                "sample_name": sample_name,
                "time_col_index": c,
                "abs_col_index": c + 1,
            })

    if not sample_pairs:
        raise ValueError("No Time(ms), Abs sample pairs found.")

    # Stop before Cary report/result/fit sections.
    # These sections are not raw graph coordinates and should not be plotted.
    # Cary exports can include a TIME FITS section after the raw graph data.
    # If TIME FITS is not skipped, Cary's fit values get plotted on top of
    # raw absorbance values, causing the "fat line" problem.
    data_start = header_row_idx + 1
    data_end = len(rows)

    STOP_SECTIONS = {"TIME FITS", "RESULTS", "RATE RESULTS"}

    for i in range(data_start, len(rows)):
        first = rows[i][0].strip() if rows[i] and str(rows[i][0]).strip() else ""
        first_upper = first.upper()

        if first_upper in STOP_SECTIONS or first_upper.startswith("RATE RESULTS"):
            data_end = i
            break

    long_records = []
    sample_counts = {p["sample_name"]: 0 for p in sample_pairs}

    for row in rows[data_start:data_end]:
        for p in sample_pairs:
            tc = p["time_col_index"]
            ac = p["abs_col_index"]
            t_ms = _to_float(row[tc]) if tc < len(row) else np.nan
            absorbance = _to_float(row[ac]) if ac < len(row) else np.nan
            # Blank cells are not plotted as points because they are not coordinates,
            # but the sample itself is still kept in the sample table.
            if np.isfinite(t_ms) and np.isfinite(absorbance):
                long_records.append({
                    "sample_index": p["sample_index"],
                    "sample_name": p["sample_name"],
                    "time_ms": t_ms,
                    "time_min": t_ms / 60000.0,
                    "absorbance_340": absorbance,
                })
                sample_counts[p["sample_name"]] += 1

    sample_info_df = pd.DataFrame(sample_pairs)
    sample_info_df["n_valid_points"] = sample_info_df["sample_name"].map(sample_counts).fillna(0).astype(int)

    long_df = pd.DataFrame(long_records)
    if not long_df.empty:
        long_df = long_df.sort_values(["sample_index", "time_min"]).reset_index(drop=True)

    return sample_info_df, long_df


class CaryNADHPerSampleSelector:
    """
    Per-sample interactive slope selector for Cary NADH A340 kinetics data.

    Each sample gets:
      - its own graph
      - two-click start/end slope selection
      - editable substrate concentration in µM
      - keep/remove checkbox

    Samples with the same concentration are grouped as replicates.
    """

    def __init__(self, sample_info_df, long_df):
        self.sample_info_df = sample_info_df.copy().reset_index(drop=True)
        self.long_df = long_df.copy()
        self.sample_names = self.sample_info_df["sample_name"].tolist()

        self.clicks = {name: [] for name in self.sample_names}
        self.fit_results = {name: None for name in self.sample_names}

        self.include_widgets = {}
        self.conc_widgets = {}
        self.slope_widgets = {}
        self.r2_widgets = {}
        self.n_widgets = {}
        self.range_widgets = {}
        self.reset_buttons = {}

        self.figures = {}
        self.status_widgets = {}

        self.slope_mode = widgets.Dropdown(
            options=[
                ("Fit all points between clicked start/end times", "fit_between"),
                ("Use only the two clicked points", "two_points"),
            ],
            value="fit_between",
            description="Slope mode:",
            layout=widgets.Layout(width="430px")
        )

        self.main_output = widgets.Output()

        self._build_table()
        self._build_graph_tabs()

    def _sample_df(self, sample_name):
        if self.long_df.empty:
            return pd.DataFrame(columns=["time_min", "absorbance_340"])
        return self.long_df[self.long_df["sample_name"] == sample_name].sort_values("time_min").copy()

    def _build_table(self):
        header_style = "font-weight:bold; border-bottom:1px solid #ccc; padding-bottom:4px;"
        rows = [
            widgets.HTML(f"<div style='{header_style}'>Use?</div>"),
            widgets.HTML(f"<div style='{header_style}'>Sample</div>"),
            widgets.HTML(f"<div style='{header_style}'>Substrate (µM)</div>"),
            widgets.HTML(f"<div style='{header_style}'>Slope (Abs/min)</div>"),
            widgets.HTML(f"<div style='{header_style}'>|Slope|</div>"),
            widgets.HTML(f"<div style='{header_style}'>R²</div>"),
            widgets.HTML(f"<div style='{header_style}'>n points</div>"),
            widgets.HTML(f"<div style='{header_style}'>Selected range</div>"),
            widgets.HTML(f"<div style='{header_style}'>Reset</div>"),
        ]

        for _, row in self.sample_info_df.iterrows():
            name = row["sample_name"]
            n_valid = int(row["n_valid_points"])

            inc = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="50px"))
            conc = widgets.Text(value="", placeholder="e.g. 250", layout=widgets.Layout(width="105px"))
            slope = widgets.HTML(value="—")
            slope_mag = widgets.HTML(value="—")
            r2 = widgets.HTML(value="—")
            npts = widgets.HTML(value="—")
            rng = widgets.HTML(value="—")
            reset_btn = widgets.Button(description="Reset", layout=widgets.Layout(width="70px"))

            self.include_widgets[name] = inc
            self.conc_widgets[name] = conc
            self.slope_widgets[name] = (slope, slope_mag)
            self.r2_widgets[name] = r2
            self.n_widgets[name] = npts
            self.range_widgets[name] = rng
            self.reset_buttons[name] = reset_btn

            reset_btn.on_click(lambda b, sample_name=name: self.reset_sample(sample_name))

            rows.extend([
                inc,
                widgets.HTML(value=f"<b>{name}</b><br><span style='color:#666'>{n_valid} plotted points</span>"),
                conc,
                slope,
                slope_mag,
                r2,
                npts,
                rng,
                reset_btn,
            ])

        self.table_grid = widgets.GridBox(
            rows,
            layout=widgets.Layout(
                grid_template_columns="60px 210px 130px 130px 110px 80px 80px 150px 80px",
                grid_gap="4px 8px",
                align_items="center",
                width="100%"
            )
        )

    def _build_graph_tabs(self):
        tab_children = []
        titles = []

        for name in self.sample_names:
            fig = go.FigureWidget()
            g = self._sample_df(name)

            if len(g) > 0:
                fig.add_scatter(
                    x=g["time_min"],
                    y=g["absorbance_340"],
                    **SLOPE_SELECTOR_RAW_TRACE_STYLE,
                )
            else:
                fig.add_scatter(x=[], y=[], **SLOPE_SELECTOR_EMPTY_RAW_TRACE_STYLE)
                fig.add_annotation(
                    text="No numeric time/absorbance points found for this sample",
                    xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False
                )

            fig.add_scatter(x=[], y=[], **SLOPE_SELECTOR_BOUNDARY_TRACE_STYLE)
            fig.add_scatter(x=[], y=[], **SLOPE_SELECTOR_REGION_TRACE_STYLE)
            fig.add_scatter(x=[], y=[], **SLOPE_SELECTOR_FIT_TRACE_STYLE)

            fig.update_layout(
                title=f"{name}: NADH absorbance at 340 nm vs time",
                xaxis_title="Time (min)",
                yaxis_title="Absorbance at 340 nm",
                **SLOPE_SELECTOR_LAYOUT_STYLE,
            )

            status = widgets.HTML(
                value="Click the start point of the straight negative-slope region, then click the end point."
            )

            # Attach click callback to raw data trace.
            fig.data[0].on_click(lambda trace, points, selector, sample_name=name: self._on_click(sample_name, points))

            self.figures[name] = fig
            self.status_widgets[name] = status

            tab_children.append(widgets.VBox([status, fig]))
            titles.append(name.replace("_340.00nm", ""))

        self.tabs = widgets.Tab(children=tab_children)
        for i, title in enumerate(titles):
            self.tabs.set_title(i, title)

    def _on_click(self, sample_name, points):
        if not points.point_inds:
            return

        g = self._sample_df(sample_name).reset_index(drop=True)
        if g.empty:
            self.status_widgets[sample_name].value = "No numeric data points are available for this sample."
            return

        idx = points.point_inds[0]
        if idx >= len(g):
            return

        x = float(g.iloc[idx]["time_min"])
        y = float(g.iloc[idx]["absorbance_340"])

        self.clicks[sample_name].append((x, y))

        fig = self.figures[sample_name]
        if len(self.clicks[sample_name]) == 1:
            with fig.batch_update():
                fig.data[1].x = [x]
                fig.data[1].y = [y]
                fig.data[2].x = []
                fig.data[2].y = []
                fig.data[3].x = []
                fig.data[3].y = []
            self.status_widgets[sample_name].value = f"Start selected at {x:.5g} min. Now click the end point."
            return

        # Use the last two clicks as start/end.
        p1, p2 = self.clicks[sample_name][-2], self.clicks[sample_name][-1]
        result = self._calculate_fit(sample_name, p1, p2)
        self.fit_results[sample_name] = result
        self.clicks[sample_name] = []
        self._draw_fit(sample_name, result)
        self._update_table_row(sample_name, result)

    def _calculate_fit(self, sample_name, p1, p2):
        x1, y1 = p1
        x2, y2 = p2
        start, end = sorted([x1, x2])
        g = self._sample_df(sample_name)
        mode = self.slope_mode.value

        if mode == "two_points":
            if x2 == x1:
                raise ValueError("The two clicked points have the same time value; cannot calculate slope.")
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - slope * x1
            region = pd.DataFrame({
                "time_min": [x1, x2],
                "absorbance_340": [y1, y2],
            }).sort_values("time_min")
            r_squared = np.nan
            stderr = np.nan
            n_points = 2
        else:
            region = g[(g["time_min"] >= start) & (g["time_min"] <= end)].copy()
            if len(region) < 2:
                # Fallback to clicked points if the selected time window somehow includes fewer than 2 raw points.
                if x2 == x1:
                    raise ValueError("Need at least two different time points to calculate a slope.")
                slope = (y2 - y1) / (x2 - x1)
                intercept = y1 - slope * x1
                region = pd.DataFrame({
                    "time_min": [x1, x2],
                    "absorbance_340": [y1, y2],
                }).sort_values("time_min")
                r_squared = np.nan
                stderr = np.nan
                n_points = 2
            else:
                lr = linregress(region["time_min"], region["absorbance_340"])
                slope = float(lr.slope)
                intercept = float(lr.intercept)
                r_squared = float(lr.rvalue ** 2)
                stderr = float(lr.stderr) if lr.stderr is not None else np.nan
                n_points = int(len(region))

        return {
            "sample_name": sample_name,
            "start_min": float(start),
            "end_min": float(end),
            "clicked_start_time_min": float(x1),
            "clicked_start_abs": float(y1),
            "clicked_end_time_min": float(x2),
            "clicked_end_abs": float(y2),
            "slope_abs_per_min": float(slope),
            "slope_magnitude_abs_per_min": abs(float(slope)),
            "intercept": float(intercept),
            "r_squared": r_squared,
            "slope_stderr": stderr,
            "n_points": n_points,
            "fit_mode": mode,
            "region_df": region[["time_min", "absorbance_340"]].copy(),
        }

    def _draw_fit(self, sample_name, result):
        fig = self.figures[sample_name]
        region = result["region_df"]
        xfit = np.array([result["start_min"], result["end_min"]], dtype=float)
        yfit = result["intercept"] + result["slope_abs_per_min"] * xfit

        with fig.batch_update():
            fig.data[1].x = [result["clicked_start_time_min"], result["clicked_end_time_min"]]
            fig.data[1].y = [result["clicked_start_abs"], result["clicked_end_abs"]]
            fig.data[2].x = region["time_min"]
            fig.data[2].y = region["absorbance_340"]
            fig.data[3].x = xfit
            fig.data[3].y = yfit

        r2_txt = "NA" if not np.isfinite(result["r_squared"]) else f"{result['r_squared']:.5f}"
        self.status_widgets[sample_name].value = (
            f"Saved slope for <b>{sample_name}</b>: "
            f"{result['slope_abs_per_min']:.6g} Abs/min, "
            f"|slope| = {result['slope_magnitude_abs_per_min']:.6g} Abs/min, "
            f"R² = {r2_txt}, n = {result['n_points']}."
        )

    def _update_table_row(self, sample_name, result):
        slope, slope_mag = self.slope_widgets[sample_name]
        slope.value = f"{result['slope_abs_per_min']:.6g}"
        slope_mag.value = f"{result['slope_magnitude_abs_per_min']:.6g}"
        self.r2_widgets[sample_name].value = "—" if not np.isfinite(result["r_squared"]) else f"{result['r_squared']:.5f}"
        self.n_widgets[sample_name].value = str(result["n_points"])
        self.range_widgets[sample_name].value = f"{result['start_min']:.5g}–{result['end_min']:.5g} min"

    def reset_sample(self, sample_name):
        self.fit_results[sample_name] = None
        self.clicks[sample_name] = []
        fig = self.figures[sample_name]
        with fig.batch_update():
            fig.data[1].x = []
            fig.data[1].y = []
            fig.data[2].x = []
            fig.data[2].y = []
            fig.data[3].x = []
            fig.data[3].y = []
        slope, slope_mag = self.slope_widgets[sample_name]
        slope.value = "—"
        slope_mag.value = "—"
        self.r2_widgets[sample_name].value = "—"
        self.n_widgets[sample_name].value = "—"
        self.range_widgets[sample_name].value = "—"
        self.status_widgets[sample_name].value = "Selection reset. Click start point, then end point."

    def _parse_concentration_uM(self, sample_name):
        raw = self.conc_widgets[sample_name].value
        if raw is None or str(raw).strip() == "":
            return np.nan
        try:
            return float(str(raw).strip())
        except ValueError:
            return np.nan

    def get_sample_results(self):
        """Return one row per sample, including keep/remove choice and entered substrate concentration."""
        records = []
        for _, info in self.sample_info_df.iterrows():
            name = info["sample_name"]
            fit = self.fit_results.get(name)
            conc = self._parse_concentration_uM(name)
            use = bool(self.include_widgets[name].value)
            base = {
                "use_for_analysis": use,
                "sample_index": int(info["sample_index"]),
                "sample_name": name,
                "substrate_conc_uM": conc,
                "n_valid_points_in_trace": int(info["n_valid_points"]),
            }
            if fit is None:
                base.update({
                    "slope_abs_per_min": np.nan,
                    "slope_magnitude_abs_per_min": np.nan,
                    "r_squared": np.nan,
                    "slope_stderr": np.nan,
                    "n_points_used_for_slope": np.nan,
                    "start_min": np.nan,
                    "end_min": np.nan,
                    "fit_mode": None,
                })
            else:
                base.update({
                    "slope_abs_per_min": fit["slope_abs_per_min"],
                    "slope_magnitude_abs_per_min": fit["slope_magnitude_abs_per_min"],
                    "r_squared": fit["r_squared"],
                    "slope_stderr": fit["slope_stderr"],
                    "n_points_used_for_slope": fit["n_points"],
                    "start_min": fit["start_min"],
                    "end_min": fit["end_min"],
                    "fit_mode": fit["fit_mode"],
                })
            records.append(base)
        return pd.DataFrame(records)

    def display(self):
        intro = widgets.HTML(
            value=(
                "<b>Step 1:</b> Enter substrate concentration in µM for each sample. "
                "<b>Step 2:</b> Keep checked only the samples you want included. "
                "<b>Step 3:</b> Use the tabs below to click start/end points on each sample graph. "
                "Replicates with the same concentration are kept as raw sample-level points for fitting; mean/error summaries are still calculated for display."
            )
        )
        display(widgets.VBox([
            intro,
            self.slope_mode,
            widgets.HTML("<h3>Sample analysis sheet</h3>"),
            self.table_grid,
            widgets.HTML("<h3>Separate slope-selection graphs</h3>"),
            self.tabs,
            self.main_output,
        ]))



def build_selected_line_export(selector):
    """
    Build two export tables from the saved graph-selector regions.

    selected_timepoints:
        One row for every raw time point used in the saved line fit.
        Includes fitted A340 and residuals.

    selected_line_calculations:
        One row per sample with the two clicked boundary points, delta values,
        direct two-point slope, saved fit slope/intercept, equation, R²,
        standard error, number of points, and selected time range.
    """
    timepoint_records = []
    calculation_records = []

    for _, info in selector.sample_info_df.iterrows():
        sample_name = info["sample_name"]
        fit = selector.fit_results.get(sample_name)

        if fit is None:
            continue

        concentration_uM = selector._parse_concentration_uM(sample_name)
        use_for_analysis = bool(selector.include_widgets[sample_name].value)

        x1 = float(fit["clicked_start_time_min"])
        y1 = float(fit["clicked_start_abs"])
        x2 = float(fit["clicked_end_time_min"])
        y2 = float(fit["clicked_end_abs"])

        delta_time_min = x2 - x1
        delta_absorbance = y2 - y1
        direct_two_point_slope = (
            delta_absorbance / delta_time_min
            if delta_time_min != 0
            else np.nan
        )

        saved_slope = float(fit["slope_abs_per_min"])
        saved_intercept = float(fit["intercept"])
        equation = (
            f"A340 = ({saved_slope:.12g} * time_min) "
            f"+ ({saved_intercept:.12g})"
        )

        calculation_records.append({
            "use_for_analysis": use_for_analysis,
            "sample_index": int(info["sample_index"]),
            "sample_name": sample_name,
            "substrate_conc_uM": concentration_uM,
            "fit_mode": fit["fit_mode"],
            "clicked_point_1_time_min": x1,
            "clicked_point_1_absorbance_340": y1,
            "clicked_point_2_time_min": x2,
            "clicked_point_2_absorbance_340": y2,
            "delta_time_min_point2_minus_point1": delta_time_min,
            "delta_absorbance_point2_minus_point1": delta_absorbance,
            "direct_two_point_slope_abs_per_min": direct_two_point_slope,
            "saved_fit_slope_abs_per_min": saved_slope,
            "saved_fit_slope_magnitude_abs_per_min": abs(saved_slope),
            "saved_fit_intercept": saved_intercept,
            "saved_fit_equation": equation,
            "r_squared": fit["r_squared"],
            "slope_stderr": fit["slope_stderr"],
            "n_points_used_for_saved_fit": int(fit["n_points"]),
            "selected_start_min": float(fit["start_min"]),
            "selected_end_min": float(fit["end_min"]),
        })

        region = (
            fit["region_df"][["time_min", "absorbance_340"]]
            .copy()
            .sort_values("time_min")
            .reset_index(drop=True)
        )

        for point_index, row in region.iterrows():
            time_min = float(row["time_min"])
            observed_absorbance = float(row["absorbance_340"])
            fitted_absorbance = saved_intercept + saved_slope * time_min
            residual = observed_absorbance - fitted_absorbance

            boundary_labels = []
            if (
                np.isclose(time_min, x1, rtol=0, atol=1e-12)
                and np.isclose(observed_absorbance, y1, rtol=0, atol=1e-12)
            ):
                boundary_labels.append("clicked_point_1")
            if (
                np.isclose(time_min, x2, rtol=0, atol=1e-12)
                and np.isclose(observed_absorbance, y2, rtol=0, atol=1e-12)
            ):
                boundary_labels.append("clicked_point_2")

            timepoint_records.append({
                "use_for_analysis": use_for_analysis,
                "sample_index": int(info["sample_index"]),
                "sample_name": sample_name,
                "substrate_conc_uM": concentration_uM,
                "fit_mode": fit["fit_mode"],
                "point_number_in_selected_region": point_index + 1,
                "time_min": time_min,
                "observed_absorbance_340": observed_absorbance,
                "fitted_absorbance_340": fitted_absorbance,
                "residual_observed_minus_fitted": residual,
                "clicked_boundary_point": ";".join(boundary_labels),
                "saved_fit_slope_abs_per_min": saved_slope,
                "saved_fit_intercept": saved_intercept,
                "saved_fit_equation": equation,
                "r_squared": fit["r_squared"],
                "selected_start_min": float(fit["start_min"]),
                "selected_end_min": float(fit["end_min"]),
            })

    selected_timepoints_df = pd.DataFrame(timepoint_records)
    selected_line_calculations_df = pd.DataFrame(calculation_records)

    return selected_timepoints_df, selected_line_calculations_df


def add_editable_excel_formulas(export_file):
    """
    Add editable Excel formulas to the two selected-line export sheets.

    Existing numerical columns remain unchanged. Separate formula columns are
    added so users can edit source cells in Excel and recalculate dependent
    values automatically.
    """
    from openpyxl import load_workbook
    from openpyxl.utils import get_column_letter

    export_path = Path(export_file)
    workbook = load_workbook(export_path)

    def header_map(worksheet):
        return {
            str(cell.value): cell.column
            for cell in worksheet[1]
            if cell.value is not None
        }

    def append_formula_column(worksheet, header, formula_builder):
        new_col = worksheet.max_column + 1
        worksheet.cell(row=1, column=new_col, value=header)
        headers = header_map(worksheet)

        for row_number in range(2, worksheet.max_row + 1):
            worksheet.cell(
                row=row_number,
                column=new_col,
                value=formula_builder(row_number, headers),
            )

    if "selected_line_calculations" in workbook.sheetnames:
        ws = workbook["selected_line_calculations"]

        append_formula_column(
            ws,
            "excel_delta_time_min_formula",
            lambda r, h: (
                f"={get_column_letter(h['clicked_point_2_time_min'])}{r}"
                f"-{get_column_letter(h['clicked_point_1_time_min'])}{r}"
            ),
        )

        append_formula_column(
            ws,
            "excel_delta_absorbance_formula",
            lambda r, h: (
                f"={get_column_letter(h['clicked_point_2_absorbance_340'])}{r}"
                f"-{get_column_letter(h['clicked_point_1_absorbance_340'])}{r}"
            ),
        )

        append_formula_column(
            ws,
            "excel_direct_two_point_slope_formula",
            lambda r, h: (
                f'=IFERROR('
                f'{get_column_letter(h["excel_delta_absorbance_formula"])}{r}/'
                f'{get_column_letter(h["excel_delta_time_min_formula"])}{r},'
                f'NA())'
            ),
        )

        append_formula_column(
            ws,
            "excel_saved_fit_slope_magnitude_formula",
            lambda r, h: (
                f"=ABS({get_column_letter(h['saved_fit_slope_abs_per_min'])}{r})"
            ),
        )

        append_formula_column(
            ws,
            "excel_fitted_absorbance_at_point_1_formula",
            lambda r, h: (
                f"={get_column_letter(h['saved_fit_intercept'])}{r}"
                f"+{get_column_letter(h['saved_fit_slope_abs_per_min'])}{r}"
                f"*{get_column_letter(h['clicked_point_1_time_min'])}{r}"
            ),
        )

        append_formula_column(
            ws,
            "excel_fitted_absorbance_at_point_2_formula",
            lambda r, h: (
                f"={get_column_letter(h['saved_fit_intercept'])}{r}"
                f"+{get_column_letter(h['saved_fit_slope_abs_per_min'])}{r}"
                f"*{get_column_letter(h['clicked_point_2_time_min'])}{r}"
            ),
        )

        append_formula_column(
            ws,
            "excel_line_equation_text_formula",
            lambda r, h: (
                f'="A340 = ("&TEXT('
                f'{get_column_letter(h["saved_fit_slope_abs_per_min"])}{r},'
                f'"0.000000000000")&" * time_min) + ("&TEXT('
                f'{get_column_letter(h["saved_fit_intercept"])}{r},'
                f'"0.000000000000")&")"'
            ),
        )

    if "selected_timepoints" in workbook.sheetnames:
        ws = workbook["selected_timepoints"]

        append_formula_column(
            ws,
            "excel_fitted_absorbance_formula",
            lambda r, h: (
                f"={get_column_letter(h['saved_fit_intercept'])}{r}"
                f"+{get_column_letter(h['saved_fit_slope_abs_per_min'])}{r}"
                f"*{get_column_letter(h['time_min'])}{r}"
            ),
        )

        append_formula_column(
            ws,
            "excel_residual_formula",
            lambda r, h: (
                f"={get_column_letter(h['observed_absorbance_340'])}{r}"
                f"-{get_column_letter(h['excel_fitted_absorbance_formula'])}{r}"
            ),
        )

    try:
        workbook.calculation.calcMode = "auto"
        workbook.calculation.fullCalcOnLoad = True
        workbook.calculation.forceFullCalc = True
    except Exception:
        pass

    workbook.save(export_path)


def michaelis_menten(S, Vmax, Km):
    """Michaelis-Menten saturation model."""
    S = np.asarray(S, dtype=float)
    return (Vmax * S) / (Km + S)


def hill_equation(S, Vmax, Khalf, nH):
    """Hill equation with apparent half-saturation constant and Hill coefficient."""
    S = np.asarray(S, dtype=float)
    return (Vmax * (S ** nH)) / ((Khalf ** nH) + (S ** nH))


def substrate_inhibition(S, Vmax, Km, Ki):
    """
    Substrate inhibition model.

    v = Vmax * S / (Km + S + S^2 / Ki)

    This is useful when the rate rises at low substrate concentration
    and drops at high substrate concentration.
    """
    S = np.asarray(S, dtype=float)
    return (Vmax * S) / (Km + S + ((S ** 2) / Ki))


KINETIC_MODEL_SPECS = {
    "Michaelis-Menten": {
        "func": michaelis_menten,
        "parameters": ["Vmax", "Km"],
        "units": ["µmol/min/mg", "µM"],
    },
    "Hill": {
        "func": hill_equation,
        "parameters": ["Vmax", "Khalf", "nH"],
        "units": ["µmol/min/mg", "µM", "unitless"],
    },
    "Substrate Inhibition": {
        "func": substrate_inhibition,
        "parameters": ["Vmax", "Km", "Ki"],
        "units": ["µmol/min/mg", "µM", "µM"],
    },
}


FIT_MODE_LABELS = {
    "raw_replicates": "Raw replicate points, no averaging/SE in fit",
    "summary_means": "Mean by substrate concentration",
}


def _safe_model_key(model_name):
    """Make a short safe name for column prefixes and sheet names."""
    return (
        model_name.lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )


def _positive_values(values):
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr) & (arr > 0)]


def build_activity_summary(
    sample_results_df,
    enzyme_conc_mg_ml,
    molecular_weight_kda,
    extinction_coeff=EXTINCTION_COEFF,
    path_length_cm=PATH_LENGTH_CM,
    reaction_volume_ml=REACTION_VOLUME_ML,
    enzyme_volume_ul=ENZYME_VOLUME_UL,
):
    """
    Calculate sample-level activity and a display summary by substrate concentration.

    Important:
    - The sample-level table keeps every replicate separately.
    - The summary table is still useful for plotting mean ± SD/SEM.
    - Final model fitting can now use raw replicate points instead of averages.
    """
    df = sample_results_df.copy()

    keep = df[
        (df["use_for_analysis"] == True) &
        df["substrate_conc_uM"].notna() &
        df["slope_magnitude_abs_per_min"].notna()
    ].copy()

    if keep.empty:
        raise ValueError(
            "No usable samples. Check that samples are selected, "
            "concentrations are entered, and slopes are chosen."
        )

    enzyme_mg_per_reaction = enzyme_conc_mg_ml * enzyme_volume_ul * 0.001

    if enzyme_mg_per_reaction <= 0:
        raise ValueError(
            "Enzyme concentration and enzyme volume must give a positive enzyme mass per reaction."
        )

    # Sample-level calculations. These raw replicate values are the default fitting input.
    keep["activity_umol_min"] = (
        keep["slope_magnitude_abs_per_min"] /
        (extinction_coeff * path_length_cm) *
        reaction_volume_ml
    )

    keep["enzyme_mg_per_reaction"] = enzyme_mg_per_reaction

    keep["specific_activity_umol_min_mg"] = (
        keep["activity_umol_min"] / enzyme_mg_per_reaction
    )

    keep["kcat_s^-1"] = (
        keep["specific_activity_umol_min_mg"] *
        molecular_weight_kda /
        60.0
    )

    grouped = (
        keep.groupby("substrate_conc_uM", as_index=False)
        .agg(**{
            "samples_used": ("sample_name", lambda x: ", ".join(map(str, x))),
            "n_replicates": ("sample_name", "count"),
            "mean_slope_abs_per_min": ("slope_magnitude_abs_per_min", "mean"),
            "sd_slope_abs_per_min": ("slope_magnitude_abs_per_min", "std"),
            "mean_r_squared": ("r_squared", "mean"),
            "activity_umol_min": ("activity_umol_min", "mean"),
            "activity_sd_umol_min": ("activity_umol_min", "std"),
            "enzyme_mg_per_reaction": ("enzyme_mg_per_reaction", "mean"),
            "specific_activity_umol_min_mg": ("specific_activity_umol_min_mg", "mean"),
            "specific_activity_sd_umol_min_mg": ("specific_activity_umol_min_mg", "std"),
            "kcat_s^-1": ("kcat_s^-1", "mean"),
            "kcat_sd_s^-1": ("kcat_s^-1", "std"),
        })
        .sort_values("substrate_conc_uM")
        .reset_index(drop=True)
    )

    grouped["sd_slope_abs_per_min"] = grouped["sd_slope_abs_per_min"].fillna(0)
    grouped["activity_sd_umol_min"] = grouped["activity_sd_umol_min"].fillna(0)
    grouped["specific_activity_sd_umol_min_mg"] = grouped["specific_activity_sd_umol_min_mg"].fillna(0)
    grouped["kcat_sd_s^-1"] = grouped["kcat_sd_s^-1"].fillna(0)

    grouped["specific_activity_sem_umol_min_mg"] = (
        grouped["specific_activity_sd_umol_min_mg"] /
        np.sqrt(grouped["n_replicates"])
    ).fillna(0)

    grouped["kcat_sem_s^-1"] = (
        grouped["kcat_sd_s^-1"] /
        np.sqrt(grouped["n_replicates"])
    ).fillna(0)

    return grouped, keep


def _get_fit_input(
    activity_summary_df,
    included_samples_df,
    fit_to="raw_replicates",
    y_col="specific_activity_umol_min_mg",
):
    """
    Return the data frame used for nonlinear fitting.

    fit_to='raw_replicates':
        each selected sample/replicate is one point.
        This does not use averages, SD, or SEM.

    fit_to='summary_means':
        one mean point per substrate concentration.
        This keeps the older behavior available for comparison.
    """
    if fit_to == "raw_replicates":
        source = included_samples_df.copy()
        source["fit_source"] = "raw_replicate"
    elif fit_to == "summary_means":
        source = activity_summary_df.copy()
        source["fit_source"] = "summary_mean"
    else:
        raise ValueError("fit_to must be 'raw_replicates' or 'summary_means'.")

    needed = ["substrate_conc_uM", y_col]
    missing = [col for col in needed if col not in source.columns]
    if missing:
        raise ValueError(f"Missing required columns for fitting: {missing}")

    fit_df = source[needed + [c for c in ["sample_name", "samples_used", "fit_source"] if c in source.columns]].copy()
    fit_df = fit_df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed)
    fit_df = fit_df[(fit_df["substrate_conc_uM"] > 0) & (fit_df[y_col] > 0)]
    fit_df = fit_df.sort_values("substrate_conc_uM").reset_index(drop=True)

    return fit_df



def _make_curve_fit_setup(model_name, fixed_params=None):
    """
    Build the function that scipy.optimize.curve_fit will actually fit.

    Updated behavior:
      - Hill: fit Vmax, Khalf, and nH from the curve.
      - Substrate Inhibition: fit Vmax, Km, and Ki from the curve.

    The fixed_params argument is kept only for backward compatibility.
    It is not used to hold nH or Ki fixed.
    """
    if model_name not in KINETIC_MODEL_SPECS:
        raise ValueError(f"Unknown model: {model_name}. Choose one of {list(KINETIC_MODEL_SPECS)}.")

    spec = KINETIC_MODEL_SPECS[model_name]
    full_names = list(spec["parameters"])
    full_units = list(spec["units"])
    func = spec["func"]

    def assemble_full_popt(free_popt):
        return np.asarray(free_popt, dtype=float)

    return {
        "fit_func": func,
        "assemble_full_popt": assemble_full_popt,
        "free_parameter_names": full_names,
        "free_parameter_units": full_units,
        "fixed_parameter_names": [],
        "fixed_parameter_values": {},
        "full_parameter_names": full_names,
        "full_parameter_units": full_units,
    }


def _initial_guesses(S, v, model_name, fixed_params=None):
    """Generate multiple starting points so nonlinear fits are less fragile."""
    S_pos = _positive_values(S)
    v_pos = _positive_values(v)

    if len(S_pos) == 0 or len(v_pos) == 0:
        raise ValueError("Fit requires positive substrate concentrations and positive rates.")

    max_v = float(np.max(v_pos))
    max_S = float(np.max(S_pos))
    min_S = float(np.min(S_pos))
    med_S = float(np.median(S_pos))

    # Substrate concentration closest to the observed maximum rate.
    peak_S = float(S[np.nanargmax(v)])

    if model_name == "Michaelis-Menten":
        guesses = [
            [max_v, med_S],
            [max_v * 1.5, min_S],
            [max_v * 2.0, max_S / 2],
        ]
        lower = [1e-12, 1e-12]
        upper = [np.inf, np.inf]

    elif model_name == "Hill":
        # Vmax, Khalf, and nH are all fitted from the curve.
        guesses = [
            [max_v, med_S, 1.0],
            [max_v * 1.5, med_S, 0.75],
            [max_v * 1.5, peak_S, 2.0],
            [max_v * 2.0, max_S / 2, 1.5],
        ]
        lower = [1e-12, 1e-12, 0.1]
        upper = [np.inf, np.inf, 8.0]

    elif model_name == "Substrate Inhibition":
        # Vmax, Km, and Ki are all fitted from the curve.
        guesses = [
            [max_v * 1.5, med_S, max_S],
            [max_v * 2.0, min_S, max_S],
            [max_v * 2.0, peak_S, max_S * 2],
            [max_v * 5.0, min_S, max_S / 2],
            [max_v * 5.0, peak_S, max_S],
        ]
        lower = [1e-12, 1e-12, 1e-12]
        upper = [np.inf, np.inf, np.inf]

    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Keep guesses inside finite lower/upper bounds.
    clean_guesses = []
    for p0 in guesses:
        p0 = np.asarray(p0, dtype=float)
        p0 = np.where(np.isfinite(p0), p0, 1.0)
        p0 = np.maximum(p0, np.asarray(lower, dtype=float) * 10)
        clean_guesses.append(p0.tolist())

    return clean_guesses, (lower, upper)


def _fit_statistics(y_obs, y_pred, n_params):
    y_obs = np.asarray(y_obs, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    residuals = y_obs - y_pred
    rss = float(np.sum(residuals ** 2))
    tss = float(np.sum((y_obs - np.mean(y_obs)) ** 2))
    r2 = 1 - rss / tss if tss != 0 else np.nan

    n = len(y_obs)

    return {
        "n_points": n,
        "n_parameters": n_params,
        "RSS": rss,
        "R_squared": r2,
    }



def fit_kinetic_model(
    activity_summary_df,
    included_samples_df,
    model_name="Michaelis-Menten",
    fit_to="raw_replicates",
    y_col="specific_activity_umol_min_mg",
    fixed_params=None,
):
    """
    Fit a kinetic model to either raw replicate data or summary means.

    Default fit_to='raw_replicates':
        every selected sample is used as its own point.
        Means, SD, and SEM are not used in the fit.

    Updated behavior:
        nH for the Hill equation and Ki for substrate inhibition are fitted
        automatically by nonlinear regression. fixed_params is kept only for
        backward compatibility and should normally be empty.
    """
    fixed_params = fixed_params or {}

    if model_name not in KINETIC_MODEL_SPECS:
        raise ValueError(f"Unknown model: {model_name}. Choose one of {list(KINETIC_MODEL_SPECS)}.")

    fit_df = _get_fit_input(
        activity_summary_df=activity_summary_df,
        included_samples_df=included_samples_df,
        fit_to=fit_to,
        y_col=y_col,
    )

    S = fit_df["substrate_conc_uM"].to_numpy(dtype=float)
    v = fit_df[y_col].to_numpy(dtype=float)

    setup = _make_curve_fit_setup(model_name, fixed_params=fixed_params)
    fit_func = setup["fit_func"]
    assemble_full_popt = setup["assemble_full_popt"]
    free_parameter_names = setup["free_parameter_names"]
    full_parameter_names = setup["full_parameter_names"]
    full_parameter_units = setup["full_parameter_units"]
    fixed_parameter_values = setup["fixed_parameter_values"]

    n_free_params = len(free_parameter_names)
    n_unique_S = fit_df["substrate_conc_uM"].nunique()

    if len(fit_df) <= n_free_params:
        raise ValueError(
            f"{model_name} has {n_free_params} fitted parameters, so it needs more than "
            f"{n_free_params} usable data points."
        )

    if n_unique_S < n_free_params:
        raise ValueError(
            f"{model_name} has {n_free_params} fitted parameters, so it needs at least "
            f"{n_free_params} unique substrate concentrations."
        )

    guesses, bounds = _initial_guesses(S, v, model_name, fixed_params=fixed_params)

    best = None
    errors = []

    for p0 in guesses:
        try:
            free_popt, pcov = curve_fit(
                fit_func,
                S,
                v,
                p0=p0,
                bounds=bounds,
                maxfev=50000,
            )
            pred = fit_func(S, *free_popt)
            rss = float(np.sum((v - pred) ** 2))

            if best is None or rss < best["rss"]:
                best = {
                    "free_popt": free_popt,
                    "pcov": pcov,
                    "rss": rss,
                    "p0": p0,
                }
        except Exception as e:
            errors.append(str(e))

    if best is None:
        raise RuntimeError(f"{model_name} fit failed from all starting guesses: {errors[:3]}")

    free_popt = np.asarray(best["free_popt"], dtype=float)
    pcov = best["pcov"]
    popt = assemble_full_popt(free_popt)

    try:
        free_perr = np.sqrt(np.diag(pcov))
    except Exception:
        free_perr = np.full(len(free_popt), np.nan)

    free_perr_by_name = {
        name: float(se) if np.isfinite(se) else np.nan
        for name, se in zip(free_parameter_names, free_perr)
    }

    pred = fit_func(S, *free_popt)
    stats = _fit_statistics(v, pred, n_params=n_free_params)

    params_rows = []
    full_value_by_name = {name: value for name, value in zip(full_parameter_names, popt)}

    for name, unit in zip(full_parameter_names, full_parameter_units):
        is_fixed = name in fixed_parameter_values
        params_rows.append({
            "model": model_name,
            "fit_to": fit_to,
            "fit_description": FIT_MODE_LABELS.get(fit_to, fit_to),
            "parameter": name,
            "value": float(full_value_by_name[name]),
            "standard_error": np.nan if is_fixed else free_perr_by_name.get(name, np.nan),
            "units": unit,
            "is_fixed": bool(is_fixed),
            "fit_role": "fixed by user" if is_fixed else "fitted by nonlinear regression",
        })

    # Append overall fit statistics as rows so the sheet stays easy to read.
    for stat_name, stat_value in stats.items():
        params_rows.append({
            "model": model_name,
            "fit_to": fit_to,
            "fit_description": FIT_MODE_LABELS.get(fit_to, fit_to),
            "parameter": stat_name,
            "value": float(stat_value) if np.isfinite(stat_value) else np.nan,
            "standard_error": np.nan,
            "units": "unitless",
            "is_fixed": False,
            "fit_role": "fit statistic",
        })

    params_df = pd.DataFrame(params_rows)

    fitted_input_df = fit_df.copy()
    model_key = _safe_model_key(model_name)
    fitted_input_df[f"{model_key}_predicted_{y_col}"] = pred
    fitted_input_df[f"{model_key}_residual_{y_col}"] = v - pred

    # full-length perr aligns with popt for annotation/export convenience.
    perr = np.array([
        np.nan if name in fixed_parameter_values else free_perr_by_name.get(name, np.nan)
        for name in full_parameter_names
    ], dtype=float)

    return {
        "model_name": model_name,
        "fit_to": fit_to,
        "y_col": y_col,
        "popt": popt,
        "perr": perr,
        "free_popt": free_popt,
        "free_perr": free_perr,
        "pcov": pcov,
        "params_df": params_df,
        "fitted_input_df": fitted_input_df,
        "stats": stats,
        "fixed_params": dict(fixed_params),
        "free_parameter_names": list(free_parameter_names),
        "fixed_parameter_values": dict(fixed_parameter_values),
    }




MODEL_DISPLAY_ORDER = ["Michaelis-Menten", "Hill", "Substrate Inhibition"]


def _get_parameter_value_and_se(fit_result, parameter_name):
    """Return fitted parameter value and standard error, or (nan, nan) if unavailable."""
    params_df = fit_result.get("params_df", pd.DataFrame())
    row = params_df[params_df["parameter"] == parameter_name]
    if row.empty:
        return np.nan, np.nan
    value = float(row["value"].iloc[0])
    se = row["standard_error"].iloc[0]
    se = float(se) if pd.notna(se) else np.nan
    return value, se


def _model_interpretation_note(
    fit_result,
    activity_summary_df,
    included_samples_df,
):
    """
    Add descriptive fit notes without choosing a winning kinetic model.

    These notes flag parameter-identifiability issues only. They do not decide
    which biochemical model is correct.
    """
    model_name = fit_result["model_name"]

    max_tested_substrate = max(
        float(activity_summary_df["substrate_conc_uM"].max()),
        float(included_samples_df["substrate_conc_uM"].max()),
    )

    if model_name == "Substrate Inhibition":
        Ki, Ki_se = _get_parameter_value_and_se(fit_result, "Ki")
        notes = []

        if np.isfinite(Ki) and max_tested_substrate > 0:
            ratio = Ki / max_tested_substrate
            if Ki > max_tested_substrate:
                notes.append(
                    f"Ki is outside the tested substrate range "
                    f"({ratio:.2f}× Smax); poorly constrained by these data"
                )

        if np.isfinite(Ki) and Ki > 0 and np.isfinite(Ki_se):
            relative_se = Ki_se / Ki
            if relative_se >= 0.50:
                notes.append(
                    f"Ki uncertainty is large ({relative_se * 100:.0f}% relative SE)"
                )

        if not notes:
            notes.append(
                "SI parameters were fit successfully; interpret inhibition from "
                "curve shape, residuals, replicate behavior, and experimental context"
            )

        return "; ".join(notes)

    if model_name == "Hill":
        nH, nH_se = _get_parameter_value_and_se(fit_result, "nH")
        if np.isfinite(nH) and abs(nH - 1.0) <= 0.10:
            return "nH is close to 1; fitted Hill behavior is near noncooperative kinetics"
        return ""

    return ""


def fit_all_kinetic_models(
    activity_summary_df,
    included_samples_df,
    fit_to="raw_replicates",
    y_col="specific_activity_umol_min_mg",
    fixed_params=None,
):
    """
    Fit MM, Hill, and substrate-inhibition models independently.

    Returns
    -------
    fit_results : dict
        Successful fit result for each model.
    comparison_df : DataFrame
        Fixed display order: MM -> Hill -> Substrate Inhibition.
        No model is selected or ranked as "best".
    """
    fixed_params = fixed_params or {}
    fit_results = {}
    rows = []

    for model_name in MODEL_DISPLAY_ORDER:
        spec = KINETIC_MODEL_SPECS[model_name]
        try:
            fit = fit_kinetic_model(
                activity_summary_df=activity_summary_df,
                included_samples_df=included_samples_df,
                model_name=model_name,
                fit_to=fit_to,
                y_col=y_col,
                fixed_params=fixed_params,
            )

            fit["interpretation_note"] = _model_interpretation_note(
                fit,
                activity_summary_df=activity_summary_df,
                included_samples_df=included_samples_df,
            )
            fit_results[model_name] = fit

            row = {
                "model": model_name,
                "fit_to": fit_to,
                "status": "fit completed",
                "n_points": fit["stats"]["n_points"],
                "n_fitted_parameters": fit["stats"]["n_parameters"],
                "RSS": fit["stats"]["RSS"],
                "R_squared": fit["stats"]["R_squared"],
                "note": fit["interpretation_note"],
            }

            for name, value in zip(spec["parameters"], fit["popt"]):
                row[name] = float(value)

            rows.append(row)

        except Exception as e:
            rows.append({
                "model": model_name,
                "fit_to": fit_to,
                "status": f"fit failed: {e}",
                "n_points": np.nan,
                "n_fitted_parameters": np.nan,
                "RSS": np.nan,
                "R_squared": np.nan,
                "note": "Fit not available",
            })

    comparison_df = pd.DataFrame(rows)
    comparison_df["model"] = pd.Categorical(
        comparison_df["model"],
        categories=MODEL_DISPLAY_ORDER,
        ordered=True,
    )
    comparison_df = comparison_df.sort_values("model").reset_index(drop=True)
    comparison_df["model"] = comparison_df["model"].astype(str)

    return fit_results, comparison_df


def compare_kinetic_models(
    activity_summary_df,
    included_samples_df,
    fit_to="raw_replicates",
    y_col="specific_activity_umol_min_mg",
    fixed_params=None,
):
    """
    Fit all available models and show them in a fixed order.

    The table reports RSS and R² for comparison but deliberately does not rank
    or choose a preferred model.
    """
    _, comparison_df = fit_all_kinetic_models(
        activity_summary_df=activity_summary_df,
        included_samples_df=included_samples_df,
        fit_to=fit_to,
        y_col=y_col,
        fixed_params=fixed_params,
    )
    return comparison_df


def add_model_predictions(
    data_df,
    model_name,
    popt,
    y_col="specific_activity_umol_min_mg",
):
    """Add model prediction and residual columns to any table with substrate_conc_uM."""
    if model_name not in KINETIC_MODEL_SPECS:
        raise ValueError(f"Unknown model: {model_name}")

    out = data_df.copy()
    func = KINETIC_MODEL_SPECS[model_name]["func"]
    model_key = _safe_model_key(model_name)

    out[f"{model_key}_predicted_{y_col}"] = func(
        out["substrate_conc_uM"].to_numpy(dtype=float),
        *popt
    )

    if y_col in out.columns:
        out[f"{model_key}_residual_{y_col}"] = (
            out[y_col] - out[f"{model_key}_predicted_{y_col}"]
        )

    return out


def _error_bar_column(error_type):
    if str(error_type).upper() == "SD":
        return "specific_activity_sd_umol_min_mg", "SD"
    return "specific_activity_sem_umol_min_mg", "SEM"




def _parameter_annotation(fit_result):
    params_df = fit_result["params_df"]
    param_names = KINETIC_MODEL_SPECS[fit_result["model_name"]]["parameters"]

    lines = [
        f"<b>{fit_result['model_name']}</b>",
        FIT_MODE_LABELS.get(fit_result["fit_to"], fit_result["fit_to"]),
    ]

    for name in param_names:
        row = params_df[params_df["parameter"] == name]
        if row.empty:
            continue
        value = float(row["value"].iloc[0])
        se = row["standard_error"].iloc[0]
        unit = row["units"].iloc[0]
        is_fixed = bool(row["is_fixed"].iloc[0]) if "is_fixed" in row.columns else False

        if is_fixed:
            lines.append(f"{name} = {value:.4g} {unit} (fixed)")
        elif pd.notna(se):
            lines.append(f"{name} = {value:.4g} ± {float(se):.2g} {unit}")
        else:
            lines.append(f"{name} = {value:.4g} {unit}")

    stats = fit_result["stats"]
    lines.extend([
        f"Fitted parameters = {stats['n_parameters']}",
        f"R² = {stats['R_squared']:.4f}",
    ])

    interpretation_note = str(fit_result.get("interpretation_note", "") or "").strip()
    if interpretation_note:
        lines.append(f"<b>Note:</b> {interpretation_note}")

    return "<br>".join(lines)


def plot_specific_activity(
    activity_summary_df,
    included_samples_df=None,
    error_type="SEM",
    show_raw_points=True,
):
    """
    Cleaner activity plot before model fitting.

    Displays raw replicate points plus optional mean ± SD/SEM summary points.
    """
    error_col, error_label = _error_bar_column(error_type)
    fig = go.Figure()

    if included_samples_df is not None and show_raw_points:
        raw_plot_df = included_samples_df.copy()

        fig.add_scatter(
            x=raw_plot_df["substrate_conc_uM"],
            y=raw_plot_df["specific_activity_umol_min_mg"],
            mode="markers",
            name="Raw replicate specific activity",
            text=raw_plot_df.get("sample_name", ""),
            customdata=np.stack([
                raw_plot_df["substrate_conc_uM"].to_numpy(dtype=float),
                raw_plot_df["slope_magnitude_abs_per_min"].to_numpy(dtype=float),
            ], axis=-1),
            marker=RAW_REPLICATE_MARKER_STYLE,
            hovertemplate=(
                "Sample: %{text}<br>"
                "Substrate: %{customdata[0]:.6g} µM<br>"
                "Slope: %{customdata[1]:.6g} Abs/min<br>"
                "Specific activity: %{y:.6g} µmol/min/mg"
                "<extra></extra>"
            ),
        )

    fig.add_scatter(
        x=activity_summary_df["substrate_conc_uM"],
        y=activity_summary_df["specific_activity_umol_min_mg"],
        mode="markers",
        name=f"Mean specific activity ± {error_label}",
        text=activity_summary_df["samples_used"],
        error_y={**MEAN_ERROR_BAR_STYLE, "array": activity_summary_df[error_col]},
        marker=MEAN_MARKER_STYLE,
        hovertemplate=(
            "Substrate: %{x} µM<br>"
            "Mean specific activity: %{y:.6g} µmol/min/mg<br>"
            f"{error_label}: %{{error_y.array:.6g}} µmol/min/mg<br>"
            "Samples: %{text}"
            "<extra></extra>"
        ),
    )

    fig.update_layout(
        title=dict(
            text=f"Specific Activity vs Substrate Concentration",
            x=0.5,
            xanchor="center",
        ),
        xaxis_title="Substrate concentration (µM)",
        yaxis_title="Specific activity (µmol/min/mg)",
        **ACTIVITY_PLOT_LAYOUT_STYLE,
    )

    fig.update_xaxes(**COMMON_PLOT_AXIS_STYLE)
    fig.update_yaxes(**COMMON_PLOT_AXIS_STYLE)

    fig.show()
    return fig


def plot_kinetic_model_fit(
    activity_summary_df,
    included_samples_df,
    fit_result,
    error_type="SEM",
    show_raw_points=True,
    show_mean_error=True,
):
    """
    Cleaner kinetic model plot.

    The fitted curve comes from the selected model.
    Raw replicate fitting remains separate from optional mean ± SE/SD display.

    Reference line:
      - Michaelis-Menten: dashed vertical line at Km
      - Hill: dashed vertical line at Khalf
      - Substrate Inhibition: dashed vertical line at Km
    """
    model_name = fit_result["model_name"]
    popt = fit_result["popt"]
    func = KINETIC_MODEL_SPECS[model_name]["func"]

    error_col, error_label = _error_bar_column(error_type)

    x_min = 0
    x_max = max(
        float(activity_summary_df["substrate_conc_uM"].max()),
        float(included_samples_df["substrate_conc_uM"].max()),
    )
    xfit = np.linspace(x_min, x_max * 1.08, 500)
    yfit = func(xfit, *popt)

    fig = go.Figure()

    if show_raw_points:
        raw_plot_df = included_samples_df.copy()

        fig.add_scatter(
            x=raw_plot_df["substrate_conc_uM"],
            y=raw_plot_df["specific_activity_umol_min_mg"],
            mode="markers",
            name="Raw replicate points",
            text=raw_plot_df.get("sample_name", ""),
            customdata=np.stack([
                raw_plot_df["substrate_conc_uM"].to_numpy(dtype=float),
                raw_plot_df["slope_magnitude_abs_per_min"].to_numpy(dtype=float),
            ], axis=-1),
            marker=RAW_REPLICATE_MARKER_STYLE,
            hovertemplate=(
                "Sample: %{text}<br>"
                "Substrate: %{customdata[0]:.6g} µM<br>"
                "Slope: %{customdata[1]:.6g} Abs/min<br>"
                "Specific activity: %{y:.6g} µmol/min/mg"
                "<extra></extra>"
            ),
        )

    if show_mean_error:
        fig.add_scatter(
            x=activity_summary_df["substrate_conc_uM"],
            y=activity_summary_df["specific_activity_umol_min_mg"],
            mode="markers",
            name=f"Mean ± {error_label} display only",
            text=activity_summary_df["samples_used"],
            error_y={**MEAN_ERROR_BAR_STYLE, "array": activity_summary_df[error_col]},
            marker=MEAN_MARKER_STYLE,
            hovertemplate=(
                "Substrate: %{x} µM<br>"
                "Mean specific activity: %{y:.6g} µmol/min/mg<br>"
                f"{error_label}: %{{error_y.array:.6g}} µmol/min/mg<br>"
                "Samples: %{text}"
                "<extra></extra>"
            ),
        )

    fig.add_scatter(
        x=xfit,
        y=yfit,
        mode="lines",
        name=f"{model_name} fit",
        line=KINETIC_FIT_LINE_STYLE,
        hovertemplate=(
            "Substrate: %{x:.6g} µM<br>"
            "Predicted specific activity: %{y:.6g} µmol/min/mg"
            "<extra></extra>"
        ),
    )

    reference_parameter = None
    if model_name in ("Michaelis-Menten", "Substrate Inhibition"):
        reference_parameter = "Km"
    elif model_name == "Hill":
        reference_parameter = "Khalf"

    if reference_parameter is not None:
        reference_value, _ = _get_parameter_value_and_se(
            fit_result,
            reference_parameter,
        )
        if np.isfinite(reference_value):
            fig.add_shape(
                type="line",
                x0=reference_value,
                x1=reference_value,
                y0=0,
                y1=1,
                xref="x",
                yref="paper",
                line={
                    "color": PAPER_BLACK,
                    "width": 1.6,
                    "dash": "dash",
                },
            )
            fig.add_annotation(
                x=reference_value,
                y=1.015,
                xref="x",
                yref="paper",
                text=f"{reference_parameter} = {reference_value:.4g} µM",
                showarrow=False,
                xanchor="center",
                yanchor="bottom",
                font={
                    "family": "Arial",
                    "size": 12,
                    "color": PAPER_BLACK,
                },
            )

    fig.add_annotation(
        text=_parameter_annotation(fit_result),
        **KINETIC_PARAMETER_ANNOTATION_POSITION,
        **KINETIC_PARAMETER_ANNOTATION_STYLE,
    )

    fig.update_layout(
        title=dict(
            text=f"{model_name} Fit of NADH Kinetics",
            x=0.5,
            xanchor="center",
        ),
        xaxis_title="Substrate concentration (µM)",
        yaxis_title="Specific activity (µmol/min/mg)",
        **KINETIC_PLOT_LAYOUT_STYLE,
    )

    fig.update_xaxes(**COMMON_PLOT_AXIS_STYLE)
    fig.update_yaxes(**COMMON_PLOT_AXIS_STYLE)

    fig.show()
    return fig



def show_model_fit_toggle(
    activity_summary_df,
    included_samples_df,
    default_model="Substrate Inhibition",
    default_fit_to="raw_replicates",
    error_type="SEM",
    fixed_params=None,
):
    """
    Interactive toggle so users can switch the fitted model and fitting input.

    Updated behavior:
      - nH is estimated from the curve when the Hill model is selected.
      - Ki is estimated from the curve when the substrate-inhibition model is selected.
    """
    fixed_params = {}

    model_widget = widgets.ToggleButtons(
        options=list(KINETIC_MODEL_SPECS.keys()),
        value=default_model if default_model in KINETIC_MODEL_SPECS else "Michaelis-Menten",
        description="Model:",
        style={"description_width": "initial"},
    )

    fit_to_widget = widgets.ToggleButtons(
        options=[
            ("Raw replicates", "raw_replicates"),
            ("Summary means", "summary_means"),
        ],
        value=default_fit_to,
        description="Fit data:",
        style={"description_width": "initial"},
    )

    raw_widget = widgets.Checkbox(
        value=True,
        description="Show raw points",
        indent=False,
    )

    mean_widget = widgets.Checkbox(
        value=True,
        description=f"Show mean ± {error_type.upper()}",
        indent=False,
    )

    output = widgets.Output()

    def _update(change=None):
        with output:
            clear_output(wait=True)
            try:
                fit_result = fit_kinetic_model(
                    activity_summary_df=activity_summary_df,
                    included_samples_df=included_samples_df,
                    model_name=model_widget.value,
                    fit_to=fit_to_widget.value,
                    fixed_params={},
                )
                print(f"Model: {model_widget.value}")
                print(f"Fit input: {FIT_MODE_LABELS.get(fit_to_widget.value, fit_to_widget.value)}")
                if model_widget.value == "Hill":
                    print("nH is fitted automatically from the curve.")
                if model_widget.value == "Substrate Inhibition":
                    print("Ki is fitted automatically from the curve.")
                display(fit_result["params_df"])
                plot_kinetic_model_fit(
                    activity_summary_df=activity_summary_df,
                    included_samples_df=included_samples_df,
                    fit_result=fit_result,
                    error_type=error_type,
                    show_raw_points=raw_widget.value,
                    show_mean_error=mean_widget.value,
                )
            except Exception as e:
                print(f"Fit could not be completed: {e}")

    for w in [model_widget, fit_to_widget, raw_widget, mean_widget]:
        w.observe(_update, names="value")

    controls = widgets.VBox([
        widgets.HTML("<h3>Interactive kinetic model fit</h3>"),
        widgets.HTML(
            "Default fit uses <b>raw replicate points</b>. "
            "Mean ± error bars are display-only unless you switch to summary means. "
            "In this version, <b>nH</b> and <b>Ki</b> are fitted automatically from the curve."
        ),
        widgets.HBox([model_widget, fit_to_widget]),
        widgets.HBox([raw_widget, mean_widget]),
    ])

    display(widgets.VBox([controls, output]))
    _update()


def run_final_analysis(
    selector,
    enzyme_conc_mg_ml,
    molecular_weight_kda,
    extinction_coeff=EXTINCTION_COEFF,
    path_length_cm=PATH_LENGTH_CM,
    reaction_volume_ml=REACTION_VOLUME_ML,
    enzyme_volume_ul=ENZYME_VOLUME_UL,
    export_file=OUTPUT_FILE,
    error_type="SEM",
    kinetic_model="Michaelis-Menten",
    fit_to="raw_replicates",
    show_fit_widget=True,
    fixed_nH=None,
    fixed_Ki_uM=None,
):
    """
    Run final analysis after the user has:
      - selected start/end slope points for each sample
      - entered substrate concentrations in µM
      - checked/unchecked samples to keep/remove

    Model behavior:
      - Michaelis-Menten, Hill, and substrate-inhibition fits are ALL generated.
      - No model is automatically selected or declared best.
      - The model comparison table stays in fixed MM -> Hill -> SI order.
      - RSS and R² are reported as fit descriptors only.
      - The `kinetic_model` argument is retained only as the default model shown
        first in the optional interactive toggle; it is not treated as a winner.
      - nH and Ki are estimated automatically from their respective curves.
    """
    sample_results_df = selector.get_sample_results()
    selected_timepoints_df, selected_line_calculations_df = build_selected_line_export(selector)

    activity_summary_df, included_samples_df = build_activity_summary(
        sample_results_df,
        enzyme_conc_mg_ml=enzyme_conc_mg_ml,
        molecular_weight_kda=molecular_weight_kda,
        extinction_coeff=extinction_coeff,
        path_length_cm=path_length_cm,
        reaction_volume_ml=reaction_volume_ml,
        enzyme_volume_ul=enzyme_volume_ul,
    )

    fixed_params = {}

    print("Sample-level slope results:")
    display(sample_results_df)

    print("Included sample-level activity values used for raw replicate fitting:")
    display(included_samples_df)

    print("Activity summary by substrate concentration, for display/error bars:")
    display(activity_summary_df)

    print("Kinetic parameters:")
    display(pd.DataFrame([
        {"parameter": "nH", "value": "fit from curve", "units": "unitless", "used_in_model": "Hill"},
        {"parameter": "Ki", "value": "fit from curve", "units": "µM", "used_in_model": "Substrate Inhibition"},
    ]))

    print("Preview plot:")
    plot_specific_activity(
        activity_summary_df=activity_summary_df,
        included_samples_df=included_samples_df,
        error_type=error_type,
        show_raw_points=True,
    )

    fit_results, comparison_df = fit_all_kinetic_models(
        activity_summary_df=activity_summary_df,
        included_samples_df=included_samples_df,
        fit_to=fit_to,
        fixed_params=fixed_params,
    )

    print("Model comparison — fixed order; no automatic winner:")
    display(comparison_df)

    print("All kinetic model fits:")
    for model_name in MODEL_DISPLAY_ORDER:
        fit_result = fit_results.get(model_name)
        if fit_result is None:
            failed_row = comparison_df[comparison_df["model"] == model_name]
            status = failed_row["status"].iloc[0] if not failed_row.empty else "fit unavailable"
            print(f"{model_name}: {status}")
            continue

        print(f"{model_name}:")
        display(fit_result["params_df"])

        plot_kinetic_model_fit(
            activity_summary_df=activity_summary_df,
            included_samples_df=included_samples_df,
            fit_result=fit_result,
            error_type=error_type,
            show_raw_points=True,
            show_mean_error=True,
        )

    # Combined parameter table for all successful models.
    params_frames = []
    fitted_input_frames = []
    raw_fitted_samples_df = included_samples_df.copy()
    summary_fitted_df = activity_summary_df.copy()

    for model_name in MODEL_DISPLAY_ORDER:
        fit_result = fit_results.get(model_name)
        if fit_result is None:
            continue

        p = fit_result["params_df"].copy()
        if "model" not in p.columns:
            p.insert(0, "model", model_name)
        p["note"] = fit_result.get("interpretation_note", "")
        params_frames.append(p)

        f = fit_result["fitted_input_df"].copy()
        if "model" not in f.columns:
            f.insert(0, "model", model_name)
        fitted_input_frames.append(f)

        raw_fitted_samples_df = add_model_predictions(
            raw_fitted_samples_df,
            model_name=model_name,
            popt=fit_result["popt"],
        )

        summary_fitted_df = add_model_predictions(
            summary_fitted_df,
            model_name=model_name,
            popt=fit_result["popt"],
        )

    params_df = (
        pd.concat(params_frames, ignore_index=True)
        if params_frames
        else pd.DataFrame()
    )
    fitted_input_df = (
        pd.concat(fitted_input_frames, ignore_index=True)
        if fitted_input_frames
        else pd.DataFrame()
    )

    if show_fit_widget:
        show_model_fit_toggle(
            activity_summary_df=activity_summary_df,
            included_samples_df=included_samples_df,
            default_model=kinetic_model,
            default_fit_to=fit_to,
            error_type=error_type,
            fixed_params=fixed_params,
        )

    curve_fitted_parameters_df = pd.DataFrame([
        {"parameter": "nH", "value": "estimated by nonlinear regression", "units": "unitless", "used_in_model": "Hill"},
        {"parameter": "Ki", "value": "estimated by nonlinear regression", "units": "µM", "used_in_model": "Substrate Inhibition"},
    ])

    with pd.ExcelWriter(export_file, engine="openpyxl") as writer:
        sample_results_df.to_excel(writer, sheet_name="sample_slope_sheet", index=False)
        selected_timepoints_df.to_excel(writer, sheet_name="selected_timepoints", index=False)
        selected_line_calculations_df.to_excel(writer, sheet_name="selected_line_calculations", index=False)
        included_samples_df.to_excel(writer, sheet_name="included_samples", index=False)
        activity_summary_df.to_excel(writer, sheet_name="activity_summary", index=False)
        curve_fitted_parameters_df.to_excel(writer, sheet_name="curve_fitted_parameters", index=False)
        comparison_df.to_excel(writer, sheet_name="model_comparison", index=False)
        params_df.to_excel(writer, sheet_name="kinetic_fit_params", index=False)
        fitted_input_df.to_excel(writer, sheet_name="fit_input_predictions", index=False)
        raw_fitted_samples_df.to_excel(writer, sheet_name="raw_fitted_samples", index=False)
        summary_fitted_df.to_excel(writer, sheet_name="summary_fitted", index=False)

    add_editable_excel_formulas(export_file)

    print(f"Exported results to: {export_file}")

    return {
        "sample_results_df": sample_results_df,
        "selected_timepoints_df": selected_timepoints_df,
        "selected_line_calculations_df": selected_line_calculations_df,
        "included_samples_df": included_samples_df,
        "activity_summary_df": activity_summary_df,
        "curve_fitted_parameters_df": curve_fitted_parameters_df,
        "fixed_params_df": curve_fitted_parameters_df,
        "fixed_params": fixed_params,
        "model_comparison_df": comparison_df,
        "all_model_fit_results": fit_results,
        "params_df": params_df,
        "fitted_input_df": fitted_input_df,
        "raw_fitted_samples_df": raw_fitted_samples_df,
        "summary_fitted_df": summary_fitted_df,
    }




## 1. Load the Cary CSV

Change the filename below if needed. The parser expects the Cary export format with repeating `Time(ms), Abs` pairs.

In [2]:
# Cary CSV input path is hardcoded here.
# Use r"..." so Windows backslashes are read correctly.
CARY_FILE = r"C:\Users\crumb\OneDrive\Desktop\kinetics notebook\rcPEPCK fresh rxn (206 mg PEP stock).csv"
sample_info_df, long_df = parse_cary_kinetics_csv(CARY_FILE)

print("Samples found in Cary file:")
display(sample_info_df)

print("First few parsed graph coordinates:")
display(long_df.head())


Samples found in Cary file:


,sample_index,sample_name,time_col_index,abs_col_index,n_valid_points
0,1,Sample 1_340.00nm,1,2,1810
1,2,Sample 2_340.00nm,3,4,1810
2,3,Sample 3_340.00nm,5,6,1810
3,4,Sample 4_340.00nm,7,8,1810
4,5,Sample 5_340.00nm,9,10,1810
5,6,Sample 6_340.00nm,11,12,1810
6,7,Sample 7_340.00nm,13,14,1810
7,8,Sample 8_340.00nm,15,16,1810
8,9,Sample 9_340.00nm,17,18,1810
9,10,Sample 10_340.00nm,19,20,1810


First few parsed graph coordinates:


,sample_index,sample_name,time_ms,time_min,absorbance_340
0,1,Sample 1_340.00nm,0.0,0.000000,0.001172
1,1,Sample 1_340.00nm,100.0,0.001667,0.001218
2,1,Sample 1_340.00nm,200.0,0.003333,0.001151
3,1,Sample 1_340.00nm,300.0,0.005000,0.000953
4,1,Sample 1_340.00nm,400.0,0.006667,0.001142


## 2. Open the per-sample slope selector

Use the table to enter substrate concentration in **µM** and choose whether to keep/remove each sample.

Then open each sample tab and click:

1. Start point of the straight negative-slope region
2. End point of the straight negative-slope region

By default, the slope is calculated by fitting all raw points between the two clicked times.

In [3]:
selector = CaryNADHPerSampleSelector(sample_info_df, long_df)
selector.display()

## 3. Preview the sample sheet

Run this after selecting slopes and entering substrate concentrations to check what will be used.

In [4]:
sample_results_df = selector.get_sample_results()
display(sample_results_df)

,use_for_analysis,sample_index,sample_name,substrate_conc_uM,n_valid_points_in_trace,slope_abs_per_min,slope_magnitude_abs_per_min,r_squared,slope_stderr,n_points_used_for_slope,start_min,end_min,fit_mode
0,True,1,Sample 1_340.00nm,25.0,1810,-0.005406,0.005406,0.674847,0.000128,857,1.588333,3.015,fit_between
1,True,2,Sample 2_340.00nm,25.0,1810,-0.005965,0.005965,0.666509,0.000148,819,1.651667,3.015,fit_between
2,True,3,Sample 3_340.00nm,25.0,1810,-0.006154,0.006154,0.561676,0.000214,649,1.935000,3.015,fit_between
3,True,4,Sample 4_340.00nm,50.0,1810,-0.012008,0.012008,0.722401,0.000299,623,1.978333,3.015,fit_between
4,True,5,Sample 5_340.00nm,50.0,1810,-0.009968,0.009968,0.994002,0.000031,624,1.976667,3.015,fit_between
5,True,6,Sample 6_340.00nm,50.0,1810,-0.009565,0.009565,0.989245,0.000048,436,2.290000,3.015,fit_between
6,True,7,Sample 7_340.00nm,100.0,1810,-0.017071,0.017071,0.997812,0.000030,710,1.833333,3.015,fit_between
7,True,8,Sample 8_340.00nm,100.0,1810,-0.010681,0.010681,0.984746,0.000050,700,1.850000,3.015,fit_between
8,True,9,Sample 9_340.00nm,100.0,1810,-0.017998,0.017998,0.799365,0.000405,498,2.186667,3.015,fit_between
9,True,10,Sample 10_340.00nm,250.0,1810,-0.033661,0.033661,0.988021,0.000133,774,1.726667,3.015,fit_between


## Selected time points and saved line calculations

This section previews the data added to the Excel export.

- `selected_timepoints` contains every time point used in each saved line, with fitted absorbance and residual.
- `selected_line_calculations` contains the two clicked points, delta values, direct two-point slope, saved line equation, slope, intercept, R², standard error, and selected range.
- The exported workbook also includes editable Excel formula columns. Editing the clicked-point, time, absorbance, slope, or intercept source cells in Excel will recalculate the connected formula cells.

In [5]:
selected_timepoints_df, selected_line_calculations_df = build_selected_line_export(selector)

print("Selected time points used for each saved line:")
display(selected_timepoints_df)

print("Calculations associated with each saved line:")
display(selected_line_calculations_df)

Selected time points used for each saved line:


,use_for_analysis,sample_index,sample_name,substrate_conc_uM,fit_mode,point_number_in_selected_region,time_min,observed_absorbance_340,fitted_absorbance_340,residual_observed_minus_fitted,clicked_boundary_point,saved_fit_slope_abs_per_min,saved_fit_intercept,saved_fit_equation,r_squared,selected_start_min,selected_end_min
0,True,1,Sample 1_340.00nm,25.0,fit_between,1,1.588333,-0.066677,-0.066193,-0.000484,clicked_point_1,-0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,1.588333,3.015
1,True,1,Sample 1_340.00nm,25.0,fit_between,2,1.590000,-0.065733,-0.066202,0.000469,,-0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,1.588333,3.015
2,True,1,Sample 1_340.00nm,25.0,fit_between,3,1.591667,-0.066151,-0.066211,0.000060,,-0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,1.588333,3.015
3,True,1,Sample 1_340.00nm,25.0,fit_between,4,1.593333,-0.066143,-0.066220,0.000077,,-0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,1.588333,3.015
4,True,1,Sample 1_340.00nm,25.0,fit_between,5,1.595000,-0.066746,-0.066229,-0.000517,,-0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,1.588333,3.015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21521,True,30,Sample 30_340.00nm,15000.0,fit_between,573,3.008333,-0.294196,-0.293452,-0.000744,,-0.179882,0.247693,A340 = (-0.17988192921 * time_min) + (0.247693...,0.999952,2.055000,3.015
21522,True,30,Sample 30_340.00nm,15000.0,fit_between,574,3.010000,-0.294572,-0.293752,-0.000820,,-0.179882,0.247693,A340 = (-0.17988192921 * time_min) + (0.247693...,0.999952,2.055000,3.015
21523,True,30,Sample 30_340.00nm,15000.0,fit_between,575,3.011667,-0.294659,-0.294051,-0.000607,,-0.179882,0.247693,A340 = (-0.17988192921 * time_min) + (0.247693...,0.999952,2.055000,3.015
21524,True,30,Sample 30_340.00nm,15000.0,fit_between,576,3.013333,-0.295024,-0.294351,-0.000673,,-0.179882,0.247693,A340 = (-0.17988192921 * time_min) + (0.247693...,0.999952,2.055000,3.015


Calculations associated with each saved line:


,use_for_analysis,sample_index,sample_name,substrate_conc_uM,fit_mode,clicked_point_1_time_min,clicked_point_1_absorbance_340,clicked_point_2_time_min,clicked_point_2_absorbance_340,delta_time_min_point2_minus_point1,...,direct_two_point_slope_abs_per_min,saved_fit_slope_abs_per_min,saved_fit_slope_magnitude_abs_per_min,saved_fit_intercept,saved_fit_equation,r_squared,slope_stderr,n_points_used_for_saved_fit,selected_start_min,selected_end_min
0,True,1,Sample 1_340.00nm,25.0,fit_between,1.588333,-0.066677,3.015,-0.073826,1.426667,...,-0.005011,-0.005406,0.005406,-0.057607,A340 = (-0.00540556913349 * time_min) + (-0.05...,0.674847,0.000128,857,1.588333,3.015
1,True,2,Sample 2_340.00nm,25.0,fit_between,1.651667,-0.067468,3.015,-0.071582,1.363333,...,-0.003018,-0.005965,0.005965,-0.056670,A340 = (-0.00596477017707 * time_min) + (-0.05...,0.666509,0.000148,819,1.651667,3.015
2,True,3,Sample 3_340.00nm,25.0,fit_between,1.935000,-0.069085,3.015,-0.075333,1.080000,...,-0.005785,-0.006154,0.006154,-0.055815,A340 = (-0.00615352729124 * time_min) + (-0.05...,0.561676,0.000214,649,1.935000,3.015
3,True,4,Sample 4_340.00nm,50.0,fit_between,1.978333,-0.071541,3.015,-0.082942,1.036667,...,-0.010998,-0.012008,0.012008,-0.047142,A340 = (-0.0120079904771 * time_min) + (-0.047...,0.722401,0.000299,623,1.978333,3.015
4,True,5,Sample 5_340.00nm,50.0,fit_between,1.976667,-0.074417,3.015,-0.086014,1.038333,...,-0.011169,-0.009968,0.009968,-0.056169,A340 = (-0.00996804561791 * time_min) + (-0.05...,0.994002,0.000031,624,1.976667,3.015
5,True,6,Sample 6_340.00nm,50.0,fit_between,2.290000,-0.067275,3.015,-0.074348,0.725000,...,-0.009756,-0.009565,0.009565,-0.045659,A340 = (-0.00956476189664 * time_min) + (-0.04...,0.989245,0.000048,436,2.290000,3.015
6,True,7,Sample 7_340.00nm,100.0,fit_between,1.833333,-0.072423,3.015,-0.091613,1.181667,...,-0.016240,-0.017071,0.017071,-0.040483,A340 = (-0.0170712991405 * time_min) + (-0.040...,0.997812,0.000030,710,1.833333,3.015
7,True,8,Sample 8_340.00nm,100.0,fit_between,1.850000,-0.077650,3.015,-0.090968,1.165000,...,-0.011432,-0.010681,0.010681,-0.058354,A340 = (-0.0106809256816 * time_min) + (-0.058...,0.984746,0.000050,700,1.850000,3.015
8,True,9,Sample 9_340.00nm,100.0,fit_between,2.186667,-0.075258,3.015,-0.088031,0.828333,...,-0.015421,-0.017998,0.017998,-0.033212,A340 = (-0.0179975482495 * time_min) + (-0.033...,0.799365,0.000405,498,2.186667,3.015
9,True,10,Sample 10_340.00nm,250.0,fit_between,1.726667,-0.076936,3.015,-0.119912,1.288333,...,-0.033358,-0.033661,0.033661,-0.018014,A340 = (-0.0336607957672 * time_min) + (-0.018...,0.988021,0.000133,774,1.726667,3.015


## 4. Run final activity, kcat, and kinetic model analysis

Enter your enzyme concentration in **mg/mL** and molecular weight in **kDa**.

Defaults used:

- NADH extinction coefficient = 6.22 mM⁻¹ cm⁻¹
- Path length = 1 cm
- Reaction volume = 1 mL
- Enzyme volume = 5 µL

This version keeps every selected sample as a **raw replicate point** for model fitting. The mean ± SD/SEM table is still calculated for display, but it is not used in the default fit.

The notebook now fits and plots **all three models independently**:

- **Michaelis-Menten** — Vmax, Km
- **Hill equation** — Vmax, Khalf, nH
- **Substrate inhibition** — Vmax, Km, Ki

No model is automatically chosen as “best.” The comparison table is always displayed in **MM → Hill → substrate inhibition** order and reports RSS/R² for comparison only. The final interpretation is left to the user.

Each fitted plot includes a dashed vertical reference line at **Km** (MM and substrate inhibition) or **Khalf** (Hill). Substrate-inhibition output is flagged when Ki lies outside the tested substrate range or is otherwise poorly constrained.

The interactive model toggle is still available after the three automatic plots.


In [6]:
enzyme_conc_mg_ml = float(input("Enter enzyme concentration in mg/mL: "))
molecular_weight_kda = float(input("Enter molecular weight in kDa: "))

# All three models are fitted automatically; this only sets the interactive widget default.
OUTPUT_FILE = r"C:\Users\crumb\OneDrive\Desktop\kinetics notebook\cary_nadh_kinetics_results_v8_auto_nH_Ki.xlsx"

results = run_final_analysis(
    selector,
    enzyme_conc_mg_ml=enzyme_conc_mg_ml,
    molecular_weight_kda=molecular_weight_kda,
    extinction_coeff=6.22,
    path_length_cm=1.0,
    reaction_volume_ml=1.0,
    enzyme_volume_ul=5.0,
    export_file=OUTPUT_FILE,
    error_type="SEM",
    kinetic_model="Michaelis-Menten",
    fit_to="raw_replicates",
    show_fit_widget=True,
)


Sample-level slope results:


,use_for_analysis,sample_index,sample_name,substrate_conc_uM,n_valid_points_in_trace,slope_abs_per_min,slope_magnitude_abs_per_min,r_squared,slope_stderr,n_points_used_for_slope,start_min,end_min,fit_mode
0,True,1,Sample 1_340.00nm,25.0,1810,-0.005406,0.005406,0.674847,0.000128,857,1.588333,3.015,fit_between
1,True,2,Sample 2_340.00nm,25.0,1810,-0.005965,0.005965,0.666509,0.000148,819,1.651667,3.015,fit_between
2,True,3,Sample 3_340.00nm,25.0,1810,-0.006154,0.006154,0.561676,0.000214,649,1.935000,3.015,fit_between
3,True,4,Sample 4_340.00nm,50.0,1810,-0.012008,0.012008,0.722401,0.000299,623,1.978333,3.015,fit_between
4,True,5,Sample 5_340.00nm,50.0,1810,-0.009968,0.009968,0.994002,0.000031,624,1.976667,3.015,fit_between
5,True,6,Sample 6_340.00nm,50.0,1810,-0.009565,0.009565,0.989245,0.000048,436,2.290000,3.015,fit_between
6,True,7,Sample 7_340.00nm,100.0,1810,-0.017071,0.017071,0.997812,0.000030,710,1.833333,3.015,fit_between
7,True,8,Sample 8_340.00nm,100.0,1810,-0.010681,0.010681,0.984746,0.000050,700,1.850000,3.015,fit_between
8,True,9,Sample 9_340.00nm,100.0,1810,-0.017998,0.017998,0.799365,0.000405,498,2.186667,3.015,fit_between
9,True,10,Sample 10_340.00nm,250.0,1810,-0.033661,0.033661,0.988021,0.000133,774,1.726667,3.015,fit_between


Included sample-level activity values used for raw replicate fitting:


,use_for_analysis,sample_index,sample_name,substrate_conc_uM,n_valid_points_in_trace,slope_abs_per_min,slope_magnitude_abs_per_min,r_squared,slope_stderr,n_points_used_for_slope,start_min,end_min,fit_mode,activity_umol_min,enzyme_mg_per_reaction,specific_activity_umol_min_mg,kcat_s^-1
0,True,1,Sample 1_340.00nm,25.0,1810,-0.005406,0.005406,0.674847,0.000128,857,1.588333,3.015,fit_between,0.000869,0.002645,0.328568,0.380044
1,True,2,Sample 2_340.00nm,25.0,1810,-0.005965,0.005965,0.666509,0.000148,819,1.651667,3.015,fit_between,0.000959,0.002645,0.362558,0.419359
2,True,3,Sample 3_340.00nm,25.0,1810,-0.006154,0.006154,0.561676,0.000214,649,1.935000,3.015,fit_between,0.000989,0.002645,0.374031,0.432630
3,True,4,Sample 4_340.00nm,50.0,1810,-0.012008,0.012008,0.722401,0.000299,623,1.978333,3.015,fit_between,0.001931,0.002645,0.729885,0.844233
4,True,5,Sample 5_340.00nm,50.0,1810,-0.009968,0.009968,0.994002,0.000031,624,1.976667,3.015,fit_between,0.001603,0.002645,0.605890,0.700813
5,True,6,Sample 6_340.00nm,50.0,1810,-0.009565,0.009565,0.989245,0.000048,436,2.290000,3.015,fit_between,0.001538,0.002645,0.581377,0.672460
6,True,7,Sample 7_340.00nm,100.0,1810,-0.017071,0.017071,0.997812,0.000030,710,1.833333,3.015,fit_between,0.002745,0.002645,1.037649,1.200214
7,True,8,Sample 8_340.00nm,100.0,1810,-0.010681,0.010681,0.984746,0.000050,700,1.850000,3.015,fit_between,0.001717,0.002645,0.649221,0.750933
8,True,9,Sample 9_340.00nm,100.0,1810,-0.017998,0.017998,0.799365,0.000405,498,2.186667,3.015,fit_between,0.002893,0.002645,1.093950,1.265335
9,True,10,Sample 10_340.00nm,250.0,1810,-0.033661,0.033661,0.988021,0.000133,774,1.726667,3.015,fit_between,0.005412,0.002645,2.046013,2.366555


Activity summary by substrate concentration, for display/error bars:


,substrate_conc_uM,samples_used,n_replicates,mean_slope_abs_per_min,sd_slope_abs_per_min,mean_r_squared,activity_umol_min,activity_sd_umol_min,enzyme_mg_per_reaction,specific_activity_umol_min_mg,specific_activity_sd_umol_min_mg,kcat_s^-1,kcat_sd_s^-1,specific_activity_sem_umol_min_mg,kcat_sem_s^-1
0,25.0,"Sample 1_340.00nm, Sample 2_340.00nm, Sample 3...",3,0.005841,0.000389,0.634344,0.000939,0.000063,0.002645,0.355053,0.023643,0.410677,0.027347,0.013650,0.015789
1,50.0,"Sample 4_340.00nm, Sample 5_340.00nm, Sample 6...",3,0.010514,0.001310,0.901883,0.001690,0.000211,0.002645,0.639051,0.079614,0.739169,0.092086,0.045965,0.053166
2,100.0,"Sample 7_340.00nm, Sample 8_340.00nm, Sample 9...",3,0.015250,0.003984,0.927308,0.002452,0.000640,0.002645,0.926940,0.242153,1.072161,0.280090,0.139807,0.161710
3,250.0,"Sample 10_340.00nm, Sample 11_340.00nm, Sample...",3,0.033793,0.001887,0.924243,0.005433,0.000303,0.002645,2.054076,0.114684,2.375881,0.132651,0.066213,0.076586
4,500.0,"Sample 13_340.00nm, Sample 14_340.00nm, Sample...",3,0.056146,0.002058,0.999326,0.009027,0.000331,0.002645,3.412725,0.125090,3.947385,0.144687,0.072221,0.083535
5,1000.0,"Sample 16_340.00nm, Sample 17_340.00nm, Sample...",3,0.092263,0.003044,0.999828,0.014833,0.000489,0.002645,5.608071,0.185038,6.486669,0.214028,0.106832,0.123569
6,2500.0,"Sample 19_340.00nm, Sample 20_340.00nm, Sample...",3,0.139420,0.009680,0.999841,0.022415,0.001556,0.002645,8.474430,0.588408,9.802091,0.680592,0.339718,0.392940
7,5000.0,"Sample 22_340.00nm, Sample 23_340.00nm, Sample...",3,0.178497,0.001163,0.998190,0.028697,0.000187,0.002645,10.849609,0.070674,12.549381,0.081746,0.040804,0.047196
8,10000.0,"Sample 25_340.00nm, Sample 26_340.00nm, Sample...",3,0.206918,0.004010,0.999840,0.033266,0.000645,0.002645,12.577125,0.243741,14.547542,0.281927,0.140724,0.162770
9,15000.0,"Sample 28_340.00nm, Sample 29_340.00nm, Sample...",3,0.184005,0.005110,0.999836,0.029583,0.000822,0.002645,11.184452,0.310610,12.936682,0.359273,0.179331,0.207426


... (8 outputs left out)

,model,fit_to,fit_description,parameter,value,standard_error,units,is_fixed,fit_role
0,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",Vmax,12.936311,0.425627,µmol/min/mg,False,fitted by nonlinear regression
1,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",Khalf,1254.654245,126.984877,µM,False,fitted by nonlinear regression
2,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",nH,1.072000,0.079248,unitless,False,fitted by nonlinear regression
3,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",n_points,30.000000,NaN,unitless,False,fit statistic
4,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",n_parameters,3.000000,NaN,unitless,False,fit statistic
5,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",RSS,7.079965,NaN,unitless,False,fit statistic
6,Hill,raw_replicates,"Raw replicate points, no averaging/SE in fit",R_squared,0.988623,NaN,unitless,False,fit statistic


Substrate Inhibition:


,model,fit_to,fit_description,parameter,value,standard_error,units,is_fixed,fit_role
0,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",Vmax,16.272204,0.991109,µmol/min/mg,False,fitted by nonlinear regression
1,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",Km,1951.842788,226.742923,µM,False,fitted by nonlinear regression
2,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",Ki,55458.462863,17606.377521,µM,False,fitted by nonlinear regression
3,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",n_points,30.000000,NaN,unitless,False,fit statistic
4,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",n_parameters,3.000000,NaN,unitless,False,fit statistic
5,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",RSS,4.637166,NaN,unitless,False,fit statistic
6,Substrate Inhibition,raw_replicates,"Raw replicate points, no averaging/SE in fit",R_squared,0.992548,NaN,unitless,False,fit statistic


Exported results to: C:\Users\crumb\OneDrive\Desktop\kinetics notebook\cary_nadh_kinetics_results_v8_auto_nH_Ki.xlsx
